# Avisos de las búsquedas de hiperparámetros (BUG-search-short-series, PR 2a)

Un escenario por aviso, con código real de skforecast (rama `fix/search-warnings`) y los
filtros por defecto de Python, es decir, lo que ve un usuario en un notebook. Los avisos se
muestran con el estilo de skforecast (paneles de rich).

Qué cambia respecto a `0.27.x`: cada aviso de los candidatos sale **una vez al final** de la
búsqueda, con el número de candidatos en que salió; los candidatos que fallan se agrupan en un
solo "Parameters skipped"; `LongTrainingWarning` sale una vez con el total de ajustes; y el
aviso de `n_iter` de la búsqueda aleatoria remite a skforecast en lugar de scikit-learn.

Los forecasters multiserie llevan `transformer_series=StandardScaler()` para que no salga
el `DataTransformationWarning` del constructor, que no es de la búsqueda.

Datos sintéticos: ventas diarias de 2023 de dos tiendas y un producto nuevo lanzado en
septiembre (29 días de historia antes del inicio de la validación).

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from skforecast.recursive import ForecasterRecursive, ForecasterRecursiveMultiSeries
from skforecast.direct import ForecasterDirect
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_forecaster_multiseries,
    grid_search_forecaster,
    bayesian_search_forecaster,
    grid_search_forecaster_multiseries,
    random_search_forecaster_multiseries,
    bayesian_search_forecaster_multiseries,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

rng = np.random.default_rng(42)
index = pd.date_range("2023-01-01", periods=365, freq="D")


def sales(n, level):
    t = np.arange(n)
    return (
        level + 10 * np.sin(2 * np.pi * t / 7)
        + 0.3 * rng.normal(0, 3, n).cumsum() + rng.normal(0, 2, n)
    )


series = {
    "store_1": pd.Series(sales(365, 100), index=index),
    "store_2": pd.Series(sales(365, 150), index=index),
    "new_product": pd.Series(sales(120, 50), index=index[-120:]),  # since 2023-09-02
}
cv = TimeSeriesFold(steps=14, initial_train_size="2023-09-30", refit=False)
{k: (v.index[0].date(), len(v)) for k, v in series.items()}

{'store_1': (datetime.date(2023, 1, 1), 365),
 'store_2': (datetime.date(2023, 1, 1), 365),
 'new_product': (datetime.date(2023, 9, 3), 120)}

## 1. Series de distinta longitud (grid multiserie)

Con `lags=35`, el producto nuevo (29 observaciones en el entrenamiento) no se puede predecir:
sus métricas no entran en las de esos candidatos, que se evalúan sobre menos puntos.

Antes: el aviso de series no predichas salía solo si el primer candidato del grid era el
afectado (dependía del orden), seguido del aviso de `n_evaluated` del PR 1. Ahora sale solo
el de `n_evaluated`, que cubre al otro: la serie que no se predice es la que falta en
`levels=[...]`, y la acción es la misma. Los candidatos se agrupan por `lags`, con su fila en
`results`.

In [2]:
results = grid_search_forecaster_multiseries(
    forecaster    = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series        = series,
    cv            = cv,
    param_grid    = {"alpha": [0.1, 1, 10]},
    lags_grid     = [7, 35],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)
results[["lags_label", "params", "mean_absolute_error__weighted_average", "n_evaluated"]]

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ 3 of 6 candidates are evaluated on fewer points than the most complete one, so their │
│ metrics are not comparable. The best candidate of the ranking is one of them, so the │
│ ranking may favor it. To compare the candidates on the same points, pass the series  │
│ listed below in `levels` or remove the candidates with larger windows. See the       │
│ column `n_evaluated` of the results.                                                 │
│                                                                                      │
│ Candidates with less than 95% of the 276 points of the most complete one:            │
│   - lags: [1, 2, ..., 35], 3 candidates (rows 0, 1, 2): 184 (66.7%)                  │
│                                                                                      │
│ Series that every candidate evaluates on the same points:                            │
│   levels=['store_1', 'store_2']                                                      │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1336188840.py:1     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

,lags_label,params,mean_absolute_error__weighted_average,n_evaluated
0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 10},2.485249,184
1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 1},2.488698,184
2,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.1},2.489228,184
3,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.1},3.090360,276
4,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1},3.090672,276
5,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 10},3.099126,276


El mismo grid en orden inverso da exactamente los mismos avisos (antes, con `lags=35`
primero salía un aviso más).

In [3]:
results = grid_search_forecaster_multiseries(
    forecaster    = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series        = series,
    cv            = cv,
    param_grid    = {"alpha": [0.1, 1, 10]},
    lags_grid     = [35, 7],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ 3 of 6 candidates are evaluated on fewer points than the most complete one, so their │
│ metrics are not comparable. The best candidate of the ranking is one of them, so the │
│ ranking may favor it. To compare the candidates on the same points, pass the series  │
│ listed below in `levels` or remove the candidates with larger windows. See the       │
│ column `n_evaluated` of the results.                                                 │
│                                                                                      │
│ Candidates with less than 95% of the 276 points of the most complete one:            │
│   - lags: [1, 2, ..., 35], 3 candidates (rows 0, 1, 2): 184 (66.7%)                  │
│                                                                                      │
│ Series that every candidate evaluates on the same points:                            │
│   levels=['store_1', 'store_2']                                                      │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1460775808.py:1     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

### 1b. Diferencia pequeña: sale el aviso de series no predichas

Un producto de temporada que termina el 2023-10-02: 28 observaciones en el entrenamiento y
solo 2 en el test. Con `lags=35` no se predice, pero los candidatos pierden 2 de 186 puntos
(menos del 5%), así que siguen siendo comparables y no sale el aviso de `n_evaluated`. Sí
sale el de series no predichas, una vez y con el recuento: es el único que dice que esa serie
no entra en las métricas.

In [4]:
series_seasonal = {
    "store_1": series["store_1"],
    "store_2": series["store_2"],
    "seasonal_product": series["new_product"].loc[:"2023-10-02"],
}
results = grid_search_forecaster_multiseries(
    forecaster    = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series        = series_seasonal,
    cv            = cv,
    param_grid    = {"alpha": [0.1, 1, 10]},
    lags_grid     = [7, 35],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)
results[["lags_label", "params", "mean_absolute_error__weighted_average", "n_evaluated"]]

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ Some levels are not predicted in folds where they have observed values, so their     │
│ metrics, and the aggregated metrics, do not include those folds (in 3 of 6           │
│ candidates). Levels never predicted have NaN metrics and are left out of the         │
│ aggregated metrics.                                                                  │
│                                                                                      │
│ Levels not predicted:                                                                │
│   - Too short for the `window_size` (35) in the training set: 'seasonal_product'     │
│ (never predicted).                                                                   │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/3162949769.py:6     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

,lags_label,params,mean_absolute_error__weighted_average,n_evaluated
0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 10},2.485249,184
1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 1},2.488698,184
2,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.1},2.489228,184
3,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 10},2.761509,186
4,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1},2.786894,186
5,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.1},2.789894,186


## 2. Bayesian multiserie

Con `lags` en el `search_space` y ventanas de 35 y 60: como en el grid, sale solo el aviso de
`n_evaluated`, con los trials agrupados por `lags`. Antes salía un aviso de series no
predichas por cada trial afectado, más el de `n_evaluated`.

In [5]:
def search_space(trial):
    return {
        "lags": trial.suggest_categorical("lags", [7, 35, 60]),
        "alpha": trial.suggest_float("alpha", 0.01, 10, log=True),
    }


results, study = bayesian_search_forecaster_multiseries(
    forecaster    = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series        = series,
    cv            = cv,
    search_space  = search_space,
    metric        = "mean_absolute_error",
    n_trials      = 10,
    random_state  = 123,
    return_best   = False,
    show_progress = False,
)
results[["trial_number", "lags", "params", "mean_absolute_error__weighted_average", "n_evaluated"]]

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ 6 of 10 trials are evaluated on fewer points than the most complete one, so their    │
│ metrics are not comparable. The best candidate of the ranking is one of them, so the │
│ ranking may favor it. To compare the candidates on the same points, pass the series  │
│ listed below in `levels` or remove the candidates with larger windows. See the       │
│ column `n_evaluated` of the results.                                                 │
│                                                                                      │
│ Trials with less than 95% of the 276 points of the most complete one:                │
│   - lags: [1, 2, ..., 35], 3 trials (5, 7, 8): 184 (66.7%)                           │
│   - lags: [1, 2, ..., 60], 3 trials (1, 4, 9): 184 (66.7%)                           │
│                                                                                      │
│ Series that every candidate evaluates on the same points:                            │
│   levels=['store_1', 'store_2']                                                      │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/448864427.py:8      │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

,trial_number,lags,params,mean_absolute_error__weighted_average,n_evaluated
0,5,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.6808799287054754},2.488887,184
1,7,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.20003386117087654},2.489170,184
2,8,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.08645371676200105},2.489236,184
3,1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 1.133676686701445},2.755024,184
4,4,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.3939877885425336},2.762082,184
5,9,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",{'alpha': 0.3202654041533764},2.762817,184
6,6,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.048393796358465295},3.090344,276
7,0,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 0.45075889673156744},3.090473,276
8,2,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1.538682881677986},3.090988,276
9,3,"[1, 2, 3, 4, 5, 6, 7]",{'alpha': 1.6367645761140057},3.091052,276


El backtesting de un candidato muestra el detalle: qué series y folds no se predicen, y por qué.

In [6]:
metrics, _ = backtesting_forecaster_multiseries(
    forecaster    = ForecasterRecursiveMultiSeries(Ridge(), lags=60, transformer_series=StandardScaler()),
    series        = series,
    cv            = cv,
    metric        = "mean_absolute_error",
    show_progress = False,
)
metrics

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ Some levels are not predicted in folds where they have observed values, so their     │
│ metrics, and the aggregated metrics, do not include those folds. Levels never        │
│ predicted have NaN metrics and are left out of the aggregated metrics.               │
│                                                                                      │
│ Levels not predicted:                                                                │
│   - Too short for the `window_size` (60) in the training set: 'new_product' (never   │
│ predicted).                                                                          │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /Users/javier.escobar/code/github/skforecast/skforecast/model_selection/_utils.py:12 │
│ 91                                                                                   │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

,levels,mean_absolute_error,n_evaluated
0,store_1,2.962794,92
1,store_2,2.549726,92
2,new_product,NaN,0
3,average,2.756260,184
4,weighted_average,2.756260,184
5,pooling,2.756260,184


## 3. Predicciones NaN por un hueco en `exog` (#1432)

`Ridge` no admite NaN: el día sin valor de `promo` en el periodo de test da predicciones NaN,
que quedan fuera de las métricas. El texto es el mismo en todos los candidatos, así que sale
una vez tal cual (antes, en bayesian, una vez por trial).

In [7]:
stores = {k: v for k, v in series.items() if k != "new_product"}
exog = pd.DataFrame({"promo": rng.integers(0, 2, len(index)).astype(float)}, index=index)
exog.loc["2023-10-20", "promo"] = np.nan

results, study = bayesian_search_forecaster_multiseries(
    forecaster    = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series        = stores,
    exog          = exog,
    cv            = cv,
    search_space  = lambda trial: {
        "lags": trial.suggest_categorical("lags", [7, 14]),
        "alpha": trial.suggest_float("alpha", 0.01, 10, log=True),
    },
    metric        = "mean_absolute_error",
    n_trials      = 10,
    random_state  = 123,
    return_best   = False,
    show_progress = False,
)

╭──────────────────────────── MissingValuesWarning ────────────────────────────╮
│ `exog` for series 'store_1' has missing values. Most of machine learning     │
│ models do not allow missing values. Prediction method may fail.              │
│                                                                              │
│ Category : skforecast.exceptions.MissingValuesWarning                        │
│ Location :                                                                   │
│ /Users/javier.escobar/code/github/skforecast/skforecast/utils/utils.py:1668  │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)    │
╰─────────��────────────────────────────────────────────────────────────────────╯
╭──────────────────────────── MissingValuesWarning ────────────────────────────╮
│ `exog` for series 'store_2' has missing values. Most of machine learning     │
│ models do not allow missing values. Prediction method may fail.              │
│                          

╭──────────────────────────── MissingValuesWarning ────────────────────────────╮
│ `exog` for series 'store_1' has missing values. Most of machine learning     │
│ models do not allow missing values. Prediction method may fail.              │
│                                                                              │
│ Category : skforecast.exceptions.MissingValuesWarning                        │
│ Location :                                                                   │
│ /Users/javier.escobar/code/github/skforecast/skforecast/utils/utils.py:1668  │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)    │
╰─────────��────────────────────────────────────────────────────────────────────╯
╭──────────────────────────── MissingValuesWarning ────────────────────────────╮
│ `exog` for series 'store_2' has missing values. Most of machine learning     │
│ models do not allow missing values. Prediction method may fail.              │
│                          

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ Some predictions are NaN at dates with observed values and are left out of the       │
│ metrics, which are calculated with fewer values than predicted (18 in total). Number │
│ of NaN predictions per series: {'store_1': 9, 'store_2': 9}. The forecaster could    │
│ not predict these dates, for example because of missing values in the last window or │
│ in `exog`.                                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/3940624775.py:5     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

**Los paneles impresos (stdout) de "`exog` for series ... has missing values" no pasan por el
registro:** salen de los procesos de joblib del backtesting (`n_jobs='auto'` usa varios
procesos con `refit=False`), que imprimen sus avisos por su cuenta y no llegan al proceso
principal. Pasa igual en `0.27.x` y en un backtesting suelto. Con `n_jobs=1`, el mismo caso:

In [8]:
results, study = bayesian_search_forecaster_multiseries(
    forecaster    = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series        = stores,
    exog          = exog,
    cv            = cv,
    search_space  = lambda trial: {
        "lags": trial.suggest_categorical("lags", [7, 14]),
        "alpha": trial.suggest_float("alpha", 0.01, 10, log=True),
    },
    metric        = "mean_absolute_error",
    n_trials      = 10,
    random_state  = 123,
    return_best   = False,
    show_progress = False,
    n_jobs        = 1,
)

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ `exog` for series 'store_1' has missing values. Most of machine learning models do   │
│ not allow missing values. Prediction method may fail.                                │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1869852172.py:1     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ `exog` for series 'store_2' has missing values. Most of machine learning models do   │
│ not allow missing values. Prediction method may fail.                                │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1869852172.py:1     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────── MissingValuesWarning ────────────────────────────────╮
│ Some predictions are NaN at dates with observed values and are left out of the       │
│ metrics, which are calculated with fewer values than predicted (18 in total). Number │
│ of NaN predictions per series: {'store_1': 9, 'store_2': 9}. The forecaster could    │
│ not predict these dates, for example because of missing values in the last window or │
│ in `exog`.                                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.MissingValuesWarning                                │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1869852172.py:1     │
│ Suppress : warnings.simplefilter('ignore', category=MissingValuesWarning)            │
╰──────────────────────────────────────────────────────────────────────────────────────╯

## 4. Candidatos que fallan: "Parameters skipped" agrupado

`l1_ratio=2` no es válido en `ElasticNet` y `lags=400` es más larga que la serie. Antes: un
`RuntimeWarning` por candidato (6), con el error repetido. Ahora: uno solo, agrupado por
error; los `lags` que fallan con todos los parámetros se resumen en una línea.

In [9]:
results = grid_search_forecaster(
    forecaster    = ForecasterRecursive(ElasticNet(), lags=7),
    y             = series["store_1"],
    cv            = cv,
    param_grid    = {"alpha": [0.1], "l1_ratio": [0.5, 2]},
    lags_grid     = [7, 14, 400],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)
results[["lags_label", "params", "mean_absolute_error"]]

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1871573214.py:1: RuntimeWarning: Parameters skipped: 4 of 6 candidates raise an error, so they are not in the results.

InvalidParameterError: The 'l1_ratio' parameter of ElasticNet must be a float in the range [0.0, 1.0]. Got 2 instead.
  - lags: [1, 2, ..., 7], params: {'alpha': 0.1, 'l1_ratio': 2}
  - lags: [1, 2, ..., 14], params: {'alpha': 0.1, 'l1_ratio': 2}

ValueError: If `initial_train_size` is an integer, it must be greater than the `window_size` of the forecaster (400) and smaller than the length of `y` (365). If it is a date, it must be within this range of the index.
  - lags: [1, 2, ..., 400] (2 candidates)
  results = grid_search_forecaster(


,lags_label,params,mean_absolute_error
0,"[1, 2, 3, 4, 5, 6, 7]","{'alpha': 0.1, 'l1_ratio': 0.5}",2.859365
1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]","{'alpha': 0.1, 'l1_ratio': 0.5}",2.890618


Si fallan todos, el mismo aviso dice que los resultados están vacíos (antes: además, "No
valid parameter combinations found. All combinations raised exceptions.").

In [10]:
results = grid_search_forecaster(
    forecaster    = ForecasterRecursive(ElasticNet(), lags=7),
    y             = series["store_1"],
    cv            = cv,
    param_grid    = {"alpha": [0.1], "l1_ratio": [2, 3]},
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)
results

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/2359028410.py:1: RuntimeWarning: Parameters skipped: 2 of 2 candidates raise an error, so the results are empty.

InvalidParameterError: The 'l1_ratio' parameter of ElasticNet must be a float in the range [0.0, 1.0]. Got 2 instead.
  - lags: [1, 2, ..., 7], params: {'alpha': 0.1, 'l1_ratio': 2}

InvalidParameterError: The 'l1_ratio' parameter of ElasticNet must be a float in the range [0.0, 1.0]. Got 3 instead.
  - lags: [1, 2, ..., 7], params: {'alpha': 0.1, 'l1_ratio': 3}
  results = grid_search_forecaster(


,lags,lags_label,params,mean_absolute_error,params


## 5. `LongTrainingWarning`: una vez, con el total de la búsqueda

Con `refit=True` y `steps=1` cada candidato reentrena el forecaster 92 veces. Antes: el mismo
aviso una vez por candidato ("The forecaster will be fit 92 times"), porque el filtro que
debía silenciarlo tenía la categoría equivocada. Ahora sale una vez, al empezar el primer
candidato (antes de que pase el tiempo), con el total de la búsqueda.

In [11]:
cv_refit = TimeSeriesFold(steps=1, initial_train_size="2023-09-30", refit=True)

results = grid_search_forecaster(
    forecaster    = ForecasterRecursive(Ridge(), lags=7),
    y             = series["store_1"],
    cv            = cv_refit,
    param_grid    = {"alpha": [0.1, 1, 10]},
    lags_grid     = [7, 14],
    metric        = "mean_absolute_error",
    return_best   = False,
    show_progress = False,
)

╭──────────────────────────────── LongTrainingWarning ─────────────────────────────────╮
│ Each candidate fits the forecaster about 92 times, so the search (6 candidates) fits │
│ it about 552 times. This can take substantial amounts of time. If not feasible, try  │
│ with `refit = False`.                                                                │
│                                                                                      │
│ Category : skforecast.exceptions.LongTrainingWarning                                 │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1178023148.py:3     │
│ Suppress : warnings.simplefilter('ignore', category=LongTrainingWarning)             │
╰──────────────────────────────────────────────────────────────────────────────────────╯

En bayesian dice "trials"; con `ForecasterDirect`, los ajustes incluyen un modelo por step.

In [12]:
results, study = bayesian_search_forecaster(
    forecaster    = ForecasterDirect(Ridge(), lags=7, steps=3),
    y             = series["store_1"],
    cv            = TimeSeriesFold(steps=3, initial_train_size="2023-09-30", refit=True),
    search_space  = lambda trial: {"alpha": trial.suggest_float("alpha", 0.01, 10, log=True)},
    metric        = "mean_absolute_error",
    n_trials      = 4,
    random_state  = 123,
    return_best   = False,
    show_progress = False,
)

╭──────────────────────────────── LongTrainingWarning ─────────────────────────────────╮
│ Each trial fits the forecaster about 93 times (31 folds * 3 estimators), so the      │
│ search (4 trials) fits it about 372 times. This can take substantial amounts of      │
│ time. If not feasible, try with `refit = False`.                                     │
│                                                                                      │
│ Category : skforecast.exceptions.LongTrainingWarning                                 │
│ Location :                                                                           │
│ /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/3675777436.py:1     │
│ Suppress : warnings.simplefilter('ignore', category=LongTrainingWarning)             │
╰──────────────────────────────────────────────────────────────────────────────────────╯

## 6. Búsqueda aleatoria con `n_iter` mayor que el espacio

Antes: el aviso de scikit-learn, que remite a `GridSearchCV`. Ahora remite a la búsqueda de
skforecast equivalente.

In [13]:
results = random_search_forecaster_multiseries(
    forecaster          = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series              = stores,
    cv                  = cv,
    param_distributions = {"alpha": [0.1, 1]},
    n_iter              = 10,
    lags_grid           = [7, 14],
    metric              = "mean_absolute_error",
    return_best         = False,
    show_progress       = False,
)
len(results)

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/3874017476.py:1: UserWarning: The total space of parameters (2) is smaller than `n_iter` (10), so the search evaluates the 2 combinations for each value of `lags_grid`. For an exhaustive search, use `grid_search_forecaster_multiseries`.
  results = random_search_forecaster_multiseries(


4

## 7. Avisos de otras librerías: agrupados igual

`ElasticNet` con `max_iter=5` no converge. El `ConvergenceWarning` de scikit-learn cambia de
texto en cada ajuste (incluye el *duality gap*), así que se agrupa por su primera frase.

In [14]:
results = grid_search_forecaster(
    forecaster    = ForecasterRecursive(ElasticNet(max_iter=5), lags=7),
    y             = series["store_1"],
    cv            = cv,
    param_grid    = {"alpha": [0.0001, 0.001, 10]},
    metric        = "mean_absolute_error",
    return_best   = True,
    show_progress = False,
)

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/1654963936.py:1: ConvergenceWarning: Objective did not converge (in 3 of 3 candidates; details of the first one). You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.143020e+03, tolerance: 2.162e+00
  results = grid_search_forecaster(


## 8. `suppress_warnings=True`

Silencia los avisos de skforecast (también el de las series cortas), pero no "Parameters
skipped": que falten candidatos en los resultados es información importante.

In [15]:
results = grid_search_forecaster_multiseries(
    forecaster        = ForecasterRecursiveMultiSeries(Ridge(), lags=7, transformer_series=StandardScaler()),
    series            = series,
    cv                = cv,
    param_grid        = {"alpha": [0.1, 1]},
    lags_grid         = [7, 35, 400],
    metric            = "mean_absolute_error",
    return_best       = False,
    show_progress     = False,
    suppress_warnings = True,
)

/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/ipykernel_99102/14282232.py:1: RuntimeWarning: Parameters skipped: 2 of 6 candidates raise an error, so they are not in the results.

ValueError: If `initial_train_size` is an integer, it must be greater than the `window_size` of the forecaster (400) and smaller than the length of the span of `series` (365). If it is a date, it must be within this range of the index.
  - lags: [1, 2, ..., 400] (2 candidates)
  results = grid_search_forecaster_multiseries(


## Fuera de este PR (PR 2b)

Con `OneStepAheadFold`, un candidato de `lags` sin datos que entrenar sigue abortando toda la
búsqueda con un `ValueError` (con `TimeSeriesFold` se salta); y en bayesian, un trial que
falla aborta el study.